In [ ]:
%pip install -U agentops

In [1]:
import agentops
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
import os
import re


from typing import TypedDict, List, Dict, Any


In [2]:
import os
from dotenv import load_dotenv

# Carregar o arquivo .env
load_dotenv(
    r"C:\Users\eugen\Documents\ProjetoIA\.env",
    override=True
)

# Verificar configurações
print("OpenAI:", bool(os.getenv("OPENAI_API_KEY")))
print("Tavily:", bool(os.getenv("TAVILY_API_KEY")))
print("AgentOps:", bool(os.getenv("AGENTOPS_API_KEY")))

OpenAI: True
Tavily: True
AgentOps: True


In [ ]:
import agentops
from dotenv import load_dotenv

# Carregar variáveis de ambiente
load_dotenv(
    r"C:\Users\eugen\Documents\ProjetoIA\.env",
    override=True
)
agentops.init(api_key=os.environ['AGENTOPS_API_KEY'], default_tags=["langgraph-agent"])

In [4]:
class AgentState(TypedDict):
    query: str
    search_results: str
    summary: str
    final_answer: str

In [5]:
# Inicializar o modelo da OpenAI
model = ChatOpenAI(model_name="gpt-4o", temperature=0)

# Inicializar o Tavily Search
from langchain_tavily import TavilySearch

search = TavilySearch(max_results=3)

In [60]:

from langchain_tavily import TavilySearch
from agentops.sdk.decorators import tool as agentops_tool

@agentops_tool(name="web_search")
def web_search(state: AgentState) -> AgentState:
    """Realiza buscas na internet utilizando Tavily."""

    try:
        # Inicializar Tavily
        tavily_search = TavilySearch(
            max_results=3,
            search_depth="advanced"
        )

        # Realizar pesquisa
        search_results = tavily_search.invoke({
            "query": state["query"]
        })

        # Extrair resultados
        if isinstance(search_results, dict):
            results = search_results.get("results", [])
        elif isinstance(search_results, list):
            results = search_results
        else:
            results = [search_results]

        # Formatar resultados
        formatted_results = "\n".join(
            f"{r.get('title', '')}: {r.get('content', '')}"
            if isinstance(r, dict) else str(r)
            for r in results
        )

        state["search_results"] = (
            formatted_results or "Nenhum resultado encontrado."
        )

    except Exception as e:
        print(f"Erro no Tavily: {e}")
        raise

    return state


In [61]:

from agentops.sdk.decorators import operation
from langchain_core.prompts import ChatPromptTemplate

# Registrar operação no AgentOps
@operation(name="summarize_results")
def summarize_results(state: AgentState) -> AgentState:
    """Resume os resultados da pesquisa utilizando LLM."""

    # Criar prompt
    prompt = ChatPromptTemplate.from_template(
        "Resuma as seguintes informações em um parágrafo conciso em português.\n\nInformações: {search_results}"
        
    )

    # Criar cadeia de processamento
    chain = prompt | model

    # Executar modelo
    summary = chain.invoke({
        "search_results": state["search_results"]
    }).content

    # Atualizar estado
    state["summary"] = summary

    return state


In [62]:
from langchain_core.prompts import ChatPromptTemplate
from agentops.sdk.decorators import operation

@operation(name="generate_answer")
def generate_answer(state: AgentState) -> dict:

    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            """
            Você é um assistente especializado em pesquisa na web.

            Responda sempre em português brasileiro.

            FORMATO OBRIGATÓRIO:

            Comece com uma introdução sobre o assunto pesquisado.

            ### Cursos Oferecidos
            - **Nome do curso**: Descrição detalhada.
            - **Nome do curso**: Descrição detalhada.

            ### Metodologia
            - **Característica**: Explicação.
            - **Característica**: Explicação.

            ### Benefícios
            - **Benefício**: Explicação.
            - **Benefício**: Explicação.

            ### Fontes
            - Não invente uma URL durante a geração da resposta e validew os links antes de mostra-los.
            - Não inventar ou exibir a fonte: Max Cursos.
            - Nome da fonte: URL.
            
            

            REGRAS:
            1. Utilize títulos Markdown.
            2. Utilize listas com marcadores (-).
            3. Destaque os nomes em negrito.
            4. Escreva descrições informativas e objetivas.
            5. Não invente cursos ou informações.
            6. Inclua somente informações confirmadas pelas fontes.
            7. Não transforme assuntos mencionados em nomes de cursos.
            8. Se uma seção não possuir informações verificáveis,
               informe que os dados não foram encontrados.
            9. Não utilize listas numeradas.
            10. Preserve os links das fontes consultadas.
            """
        ),
        (
            "human",
            """
            Pergunta:
            {query}

            Resumo:
            {summary}

            Resultados da pesquisa:
            {search_results}
            """
        )
    ])

    chain = prompt | model

    response = chain.invoke({
        "query": state["query"],
        "summary": state["summary"],
        "search_results": state["search_results"]
    })

    return {"final_answer": response.content}

In [63]:
from langgraph.graph import StateGraph, START, END

# Construir workflow
workflow = StateGraph(AgentState)

workflow.add_node("web_search", web_search)
workflow.add_node("summarize_results", summarize_results)
workflow.add_node("generate_answer", generate_answer)

workflow.add_edge(START, "web_search")
workflow.add_edge("web_search", "summarize_results")
workflow.add_edge("summarize_results", "generate_answer")
workflow.add_edge("generate_answer", END)

graph = workflow.compile()

In [64]:

import agentops

def execute_workflow(query: str) -> str:
    """Executa o workflow e registra o rastreamento no AgentOps."""

    # Definir estado inicial
    initial_state = AgentState(
        query=query,
        search_results="",
        summary="",
        final_answer=""
    )

    # Iniciar rastreamento do AgentOps
    trace_context = agentops.start_trace(
        "langgraph-workflow",
        tags=["langgraph-agent"]
    )

    if trace_context is None:
        raise RuntimeError("AgentOps não foi inicializado corretamente.")

    try:
        # Executar o grafo
        final_state = graph.invoke(initial_state)

        # Verificar os campos retornados
        print("Campos retornados:", list(final_state.keys()))

        # Verificar resposta final
        if not final_state.get("final_answer"):
            raise RuntimeError(
                "O nó generate_answer não produziu uma resposta final."
            )

        # Retornar resposta final
        return final_state["final_answer"]

    except Exception:
        # Encerrar rastreamento com erro
        agentops.end_trace(trace_context, end_state="Error")
        raise

    finally:
        # Encerrar rastreamento com sucesso somente se não houve erro
        pass


In [65]:

# Exemplo de consulta
query = "Quais são os cursos da scoras academy?"

# Executar fluxo de trabalho
answer = execute_workflow(query)

# Exibir resposta final
print("Resposta Final:\n")
print(answer)

🖇 AgentOps: Session Replay for langgraph-workflow trace: https://app.agentops.ai/sessions?trace_id=c85872a02dd94ca79fd2c1bb7000ca41


Campos retornados: ['query', 'search_results', 'summary', 'final_answer']
Resposta Final:

A Scoras Academy, liderada por Anderson Amaral, é uma plataforma de ensino focada em Inteligência Artificial, oferecendo uma formação abrangente e prática para quem deseja se aprofundar na área. O Pacote Fix 16 é um dos destaques, proporcionando acesso vitalício a uma série de cursos que cobrem desde os fundamentos até técnicas avançadas em IA.

### Cursos Oferecidos
- **Python para Iniciantes em Inteligência Artificial**: Curso introdutório que ensina a base da programação com foco em IA, ideal para quem nunca codificou.
- **Introdução a Large Language Models (LLMs)**: Aborda o que são os modelos de linguagem, sua arquitetura técnica, incluindo tokenização, embeddings, inferência e fine-tuning.
- **Fundamentos Técnicos de LLMs**: Explora em profundidade a arquitetura técnica dos LLMs.
- **Projetos Práticos de IA**: Envolve a construção de agentes reais utilizando frameworks modernos como LangGra